# Weeks 3+ — Working with the full release (~79M rows) without downloading 79M rows

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SriyaHarshini19/PixelToPrediction/blob/main/notebooks/03_working_with_the_full_release.ipynb?flush_cache=true)

Notebooks 01–02 used the small starter CSV that ships with this repo. Your lane and capstone work
run on the **full pseudonymized warehouse release**: ~17 months of daily search performance for
~70 clients, plus a query-level table. It is hosted as Parquet on Hugging Face, and the trick of
this notebook is that you **never download or load the whole thing** — DuckDB reads only the
columns and partitions your SQL touches.

By the end you will have:
1. Connected DuckDB to the hosted release and listed every table.
2. Pulled a **feature table you designed** (aggregates per content item) into pandas.
3. Trained a quick scikit-learn model on features you built from 79M rows — on a free Colab CPU.

**Before you start (one-time, ~2 minutes):**
1. Create a free [Hugging Face account](https://huggingface.co/join).
2. Open the dataset page ([`FlyRank/internship-warehouse`](https://huggingface.co/datasets/FlyRank/internship-warehouse)) and **request access** (instant after you accept the data-use terms). **Accept the terms in your browser first — the token below 401s until access is granted (usually instant).**
3. Create a **read** token at [Settings → Access Tokens](https://huggingface.co/settings/tokens). **Never paste the token into a code cell** — your repo is public; use the `getpass` prompt below (or Colab's 🔑 Secrets panel).


In [ ]:
%pip -q install duckdb huggingface_hub


In [ ]:
import os, getpass

# Token order: env var -> Colab Secret -> prompt (last resort).
# Use a Colab Secret named HF_TOKEN (the key panel on the left) so the prompt never
# fires: if Colab reconnects while a getpass prompt is open, the kernel waits on it
# forever ('Resuming execution...') and you have to restart the runtime.
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')


## 1. Connect DuckDB to the release

DuckDB speaks `hf://` natively. The secret below authenticates every query; after that the
release behaves like a set of local tables.


In [ ]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')


That count over the daily fact touched **Parquet metadata, not data** — it finished in seconds
even though the table has ~79M rows. That is the whole workflow: push the heavy lifting into
DuckDB SQL, bring only small results into pandas.

## 2. Know your panel before you model it

History depth **differs per client** (an *unbalanced panel*). `dim_clients` tells you exactly
what each client has — check it before designing any time window.


In [ ]:
clients = con.sql(f"""
    SELECT client_hash_id, access_profile, gsc_data_start, ga4_data_start
    FROM {TABLES['dim_clients']}
    ORDER BY gsc_data_start NULLS LAST
""").df()

print('clients with 12+ months of GSC history:',
      (clients['gsc_data_start'] <= clients['gsc_data_start'].dropna().max() - __import__('pandas').Timedelta(days=365)).sum())
clients.head(10)


## 3. Build features with SQL, not with RAM

The pattern for every lane: **aggregate per content item inside DuckDB**, then hand the small
result to pandas/sklearn. Here: momentum features from the last 60 days of the panel.

**This is the heaviest cell in the notebook — expect 2–6 minutes on Colab.** It downloads ~2 months of column data over the network (RAM stays tiny; that's the point). If it runs past ~10 minutes or errors with `HTTP 429`, re-run this section against `TABLES['fact_daily_sample']` and save the full table for your final pass.


In [ ]:
features = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']}
    ),
    windowed AS (
        SELECT f.client_hash_id, f.content_hash_id,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last30,
               SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev30,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_clicks ELSE 0 END)      AS clk_last30,
               AVG(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_avg_position END)       AS pos_last30
        FROM {TABLES['fact_daily']} f, bounds b
        WHERE f.report_date > b.end_d - INTERVAL 60 DAY
        GROUP BY 1, 2
        HAVING imp_prev30 >= 100
    )
    SELECT * FROM windowed
""").df()

print(f'{len(features):,} content items with enough history')
features.head()


## 4. Add query-level signals

`fact_content_query_90d` describes **how a page earns its impressions**: across how many
distinct queries, how concentrated, how much sits in the rare/anonymized tail. One page ranking
for 40 queries is a different animal from one page ranking for 2.


In [ ]:
qsignals = con.sql(f"""
    SELECT content_hash_id,
           ANY_VALUE(content_visible_query_count)     AS visible_queries,
           ANY_VALUE(rare_impressions_share)          AS rare_share,
           ANY_VALUE(anonymized_impressions_share)    AS anon_share,
           MAX(impressions_90d)                       AS top_query_impressions,
           SUM(impressions_90d)                       AS kept_impressions
    FROM {TABLES['fact_query_90d']}
    GROUP BY content_hash_id
""").df()

qsignals['top_query_share'] = qsignals['top_query_impressions'] / qsignals['kept_impressions']
data = features.merge(qsignals, on='content_hash_id', how='left')
print(f'joined: {len(data):,} rows')
data.head()


## 5. A first honest model

Same shape as notebook 02: define a label, hold out data, compare against a dumb baseline.
Label: *did impressions decline by more than 20% month-over-month?* — built only from columns
that exist **before** the window we predict. (Momentum features from the last 30 days predicting
a label defined on those same 30 days would be leakage — so here the features come from the
prev-30 window and query-mix, and the label from the last-30 outcome.)


In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

data['is_declining'] = (data['imp_last30'] < 0.8 * data['imp_prev30']).astype(int)

feature_cols = ['imp_prev30', 'visible_queries', 'rare_share', 'anon_share', 'top_query_share']
model_data = data.dropna(subset=feature_cols)
X, y = model_data[feature_cols], model_data['is_declining']

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr, y_tr)

print(f'base rate (always predict majority): {max(y_te.mean(), 1 - y_te.mean()):.3f}')
print(classification_report(y_te, model.predict(X_te), digits=3))


Whatever number you just got: interrogate it before you believe it. Which feature carries the
signal? Does it survive a per-client split (train on some clients, test on others)? That
question — *does it generalize across clients?* — is exactly what separates a capstone-grade
result from a lucky split.

## Your turn

1. Re-run section 3 with a **90-day** window and a `HAVING` threshold of your choice.
2. Add one feature you believe in (position volatility? weekend share? query concentration?).
3. Replace the random split with **GroupShuffleSplit on `client_hash_id`** and compare.

## Working locally instead

```python
from huggingface_hub import snapshot_download
path = snapshot_download(repo_id='FlyRank/internship-warehouse', repo_type='dataset',
                         allow_patterns=['dim_*.parquet', 'fact_content_query_90d.parquet',
                                         'fact_content_daily_performance/month=2026-0*/*.parquet'])
```
Then point `REL` at that local path. Download only the month partitions you need — the
`allow_patterns` filter above is the whole trick.

---

**Where this fits:** every lane brief assumes you can produce per-content feature tables like
the one you just built. The lane datasets under the `lanes` HF repo are pre-cut examples of
exactly this pattern — but for the capstone, features you engineered yourself from the full
release beat any pre-cut file.


In [ ]:
import numpy as np # Program 1: Jacobi Iteration Method Predefined print("="*50) print("PROGRAM 1: Jacobi Iteration Method") print("="*50) def jacobi_method(A, b, x0=None, tol=1e-6, max_iter=100): n = len(A) if x0 is None: x0 = np.zeros(n) x = x0.copy() print(f"Iteration details:") print(f"{'Iter':<6} {'x1':<12} {'x2':<12} {'x3':<12} {'Error':<12}") print("-"*60)  for it in range(max_iter):         x_new = np.zeros(n)          for i in range(n):             sum1 = 0             for j in range(n):                 if j != i:                     sum1 += A[i][j] * x[j]             x_new[i] = (b[i] - sum1) / A[i][i]          error = np.linalg.norm(x_new - x, np.inf)          if n == 2:             print(f"{it+1:<6} {x_new[0]:<12.6f} {x_new[1]:<12.6f} {'':<12} {error:<12.6f}")         else:             print(f"{it+1:<6} {x_new[0]:<12.6f} {x_new[1]:<12.6f} {x_new[2]:<12.6f} {error:<12.6f}")          if error < tol:             return x_new, it+1          x = x_new.copy()      return x, max_iter  # Test with a 3x3 system A = np.array([[10, -1, 2],               [-1, 11, -1],               [2, -1, 10]], dtype=float) b = np.array([6, 25, -11], dtype=float)  print("System of equations:") print("10x₁ - x₂ + 2x₃ = 6") print("-x₁ + 11x₂ - x₃ = 25") print("2x₁ - x₂ + 10x₃ = -11") print()  solution, iterations = jacobi_method(A, b, x0=np.zeros(3))  print(f"\nSolution found after {iterations} iterations:") print(f"x₁ = {solution[0]:.8f}") print(f"x₂ = {solution[1]:.8f}") print(f"x₃ = {solution[2]:.8f}")  # Verify print("\nVerification (Ax = b):") Ax = A @ solution print(f"A*x = {Ax}") print(f"b   = {b}") print(f"Error = {np.linalg.norm(Ax - b):.8f}")

In [ ]:
import numpy as np #PROGRAM-2: Jacobi Iteration Method user defined print("="*50) print("PROGRAM-2: Jacobi Iteration Method") print("="*50)  def jacobi_method(A, b, x0, tol, max_iter):     n = len(A)     x = x0.copy()      print(f"\n{'Iter':<6} ", end="")     for i in range(n):         print(f"x{i+1}:<12", end="")     print(f"{'Error':<12}")     print("-" * (6 + 12*n + 12))      for it in range(max_iter):         x_new = np.zeros(n)          for i in range(n):             sum1 = 0             for j in range(n):                 if j != i:                     sum1 += A[i][j] * x[j]             x_new[i] = (b[i] - sum1) / A[i][i]          error = np.linalg.norm(x_new - x, np.inf)          print(f"{it+1:<6} ", end="")         for i in range(n):             print(f"{x_new[i]:<12.6f}", end="")         print(f"{error:<12.6f}")          if error < tol:             return x_new, it+1          x = x_new.copy()      return x, max_iter # User Input print("\nEnter the system of equations") n = int(input("Enter number of equations (n): "))  print("\nEnter matrix A row by row:") A = [] for i in range(n):     row = list(map(float, input(f"Row {i+1}: ").split()))     A.append(row) A = np.array(A)  print("\nEnter vector b:") b = list(map(float, input("b: ").split())) b = np.array(b)  print("\nEnter initial guess (x0):") x0 = list(map(float, input("x0: ").split())) x0 = np.array(x0)  tol = float(input("\nEnter tolerance (e.g., 1e-6): ")) max_iter = int(input("Enter max iterations: "))  print("\nSystem of equations:") for i in range(n):     eq = ""     for j in range(n):         if j == 0:             eq += f"{A[i][j]:.1f}x{j+1}"         else:             if A[i][j] >= 0:                 eq += f" + {A[i][j]:.1f}x{j+1}"             else:                 eq += f" - {abs(A[i][j]):.1f}x{j+1}"     eq += f" = {b[i]:.1f}"     print(eq)  solution, iterations = jacobi_method(A, b, x0, tol, max_iter)  print(f"\nSolution found after {iterations} iterations:") for i in range(n):     print(f"x{i+1} = {solution[i]:.8f}")  # Verify print("\nVerification (Ax = b):") Ax = A @ solution for i in range(n):     print(f"A*x[{i}] = {Ax[i]:.8f}, b[{i}] = {b[i]:.8f}") print(f"Error = {np.linalg.norm(Ax - b):.8f}")

In [ ]:
import numpy as np  # Program 3: Gauss-Seidel Iteration Method print("="*50) print("PROGRAM 3: Gauss-Seidel Iteration Method") print("="*50)  def gauss_seidel_method(A, b, x0=None, tol=1e-6, max_iter=100):     n = len(A)     if x0 is None:         x0 = np.zeros(n)      x = x0.copy()      print(f"Iteration details:")     print(f"{'Iter':<6} {'x1':<12} {'x2':<12} {'x3':<12} {'Error':<12}")     print("-"*60)      for it in range(max_iter):         x_old = x.copy()          for i in range(n):             sum1 = 0             for j in range(n):                 if j != i:                     sum1 += A[i][j] * x[j]             x[i] = (b[i] - sum1) / A[i][i]          error = np.linalg.norm(x - x_old, np.inf)          print(f"{it+1:<6} {x[0]:<12.6f} {x[1]:<12.6f} {x[2]:<12.6f} {error:<12.6f}")          if error < tol:             return x, it+1      return x, max_iter  # Test with a 3x3 system A = np.array([[10, -1, 2],               [-1, 11, -1],               [2, -1, 10]], dtype=float) b = np.array([6, 25, -11], dtype=float)  print("System of equations:") print("10x₁ - x₂ + 2x₃ = 6") print("-x₁ + 11x₂ - x₃ = 25") print("2x₁ - x₂ + 10x₃ = -11") print()  solution, iterations = gauss_seidel_method(A, b, x0=np.zeros(3))  print(f"\nSolution found after {iterations} iterations:") print(f"x₁ = {solution[0]:.8f}") print(f"x₂ = {solution[1]:.8f}") print(f"x₃ = {solution[2]:.8f}")  # Verify print("\nVerification (Ax = b):") Ax = A @ solution print(f"A*x = {Ax}") print(f"b   = {b}") print(f"Error = {np.linalg.norm(Ax - b):.8f}")

In [ ]:
import numpy as np #PROGRAM-4: Gauss-Seidel Iteration Method user defined print("="*50) print("PROGRAM-4: Gauss-Seidel Iteration Method") print("="*50)  def gauss_seidel_method(A, b, x0, tol, max_iter):     n = len(A)     x = x0.copy()      print(f"\n{'Iter':<6} ", end="")     for i in range(n):         print(f"x{i+1}:<12", end="")     print(f"{'Error':<12}")     print("-" * (6 + 12*n + 12))      for it in range(max_iter):         x_old = x.copy()          for i in range(n):             sum1 = 0 for j in range(n):                 if j != i:                     sum1 += A[i][j] * x[j]             x[i] = (b[i] - sum1) / A[i][i]          error = np.linalg.norm(x - x_old, np.inf)          print(f"{it+1:<6} ", end="")         for i in range(n):             print(f"{x[i]:<12.6f}", end="")         print(f"{error:<12.6f}")          if error < tol:             return x, it+1      return x, max_iter  # User Input print("\nEnter the system of equations") n = int(input("Enter number of equations (n): "))  print("\nEnter matrix A row by row:") A = [] for i in range(n):     row = list(map(float, input(f"Row {i+1}: ").split()))     while len(row) != n:         print(f"Error: Expected {n} numbers, got {len(row)}")         row = list(map(float, input(f"Row {i+1} (enter {n} numbers): ").split()))     A.append(row) A = np.array(A)  print("\nEnter vector b:") b = list(map(float, input("b: ").split())) while len(b) != n:     print(f"Error: Expected {n} numbers, got {len(b)}")     b = list(map(float, input(f"b (enter {n} numbers): ").split())) b = np.array(b)  print("\nEnter initial guess (x0):") x0 = list(map(float, input("x0: ").split())) while len(x0) != n:     print(f"Error: Expected {n} numbers, got {len(x0)}")     x0 = list(map(float, input(f"x0 (enter {n} numbers): ").split())) x0 = np.array(x0)  tol = float(input("\nEnter tolerance (e.g., 1e-6): ")) max_iter = int(input("Enter max iterations: ")) print("\nSystem of equations:") for i in range(n):     eq = ""     for j in range(n):         if j == 0:             eq += f"{A[i][j]:.1f}x{j+1}"         else:             if A[i][j] >= 0:                 eq += f" + {A[i][j]:.1f}x{j+1}"             else:                 eq += f" - {abs(A[i][j]):.1f}x{j+1}"     eq += f" = {b[i]:.1f}"     print(eq)  solution, iterations = gauss_seidel_method(A, b, x0, tol, max_iter)  print(f"\nSolution found after {iterations} iterations:") for i in range(n):     print(f"x{i+1} = {solution[i]:.8f}")  # Verify print("\nVerification (Ax = b):") Ax = A @ solution for i in range(n):     print(f"A*x[{i}] = {Ax[i]:.8f}, b[{i}] = {b[i]:.8f}") print(f"Error = {np.linalg.norm(Ax - b):.8f}")

In [ ]:
import numpy as np  print("="*50) print("PROGRAM-5: Gauss Elimination Method") print("="*50)  n = int(input("Enter number of equations: ")) A = np.array([list(map(float, input(f"Row {i+1}: ").split())) for i in range(n)]) b = np.array(list(map(float, input("Enter b: ").split())))  # Forward Elimination Aug = np.hstack([A, b.reshape(-1, 1)]).astype(float) for i in range(n):     for j in range(i+1, n):         factor = Aug[j][i] / Aug[i][i] Aug[j] = Aug[j] - factor * Aug[i]  # Back Substitution x = np.zeros(n) for i in range(n-1, -1, -1):     x[i] = (Aug[i][-1] - np.dot(Aug[i][:-1], x)) / Aug[i][i]  print(f"\nSolution: {x}") print(f"Verification: {np.linalg.norm(A @ x - b):.8f}")

In [ ]:
import numpy as np #PROGRAM-6: Error Tracking for both models A = np.array([[10, -1, 2],               [-1, 11, -1],               [2, -1, 10]], float) b = np.array([6, 25, -11], float) exact = np.linalg.solve(A, b)  # Jacobi with error tracking x = np.zeros(3) print("Jacobi Method:") for k in range(10):     x_new = np.zeros(3)     for i in range(3):         s = sum(A[i][j]*x[j] for j in range(3) if j != i)         x_new[i] = (b[i] - s) / A[i][i]     err = np.linalg.norm(x_new - exact, np.inf)     print(f"  Iter {k+1}: error = {err:.6f}")     x = x_new  # Gauss-Seidel with error tracking y = np.zeros(3) print("\nGauss-Seidel Method:") for k in range(10):     y_old = y.copy()     for i in range(3):         s = sum(A[i][j]*y[j] for j in range(3) if j != i)         y[i] = (b[i] - s) / A[i][i]     err = np.linalg.norm(y - exact, np.inf)     print(f"  Iter {k+1}: error = {err:.6f}")     if np.allclose(y, y_old, atol=1e-6): break